# LeetCode 736 — Parse Lisp Expression bằng Python

- Link: https://leetcode.com/problems/parse-lisp-expression/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, Phân tích cú pháp (parsing), Đệ quy (recursive descent), Ngăn xếp phạm vi (scope stack), Hash Map, Cây cú pháp (AST)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def evaluate(self, expression: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bài này yêu cầu ta viết một **trình thông dịch (interpreter) tí hon** cho một ngôn ngữ giống Lisp, chỉ có số nguyên, biến, phép cộng `add`, phép nhân `mult` và lệnh gán cục bộ `let`. Ý tưởng then chốt không nằm ở phép tính, mà ở **phạm vi biến (scope)**: một biến khai báo trong `let` bên trong sẽ **che** (shadow) biến cùng tên bên ngoài, và khi ra khỏi `let` thì giá trị cũ "sống lại".

Sau notebook này bạn sẽ:

1. Biết **tách token** (tokenize) một biểu thức có ngoặc.
2. Viết lời giải baseline đệ quy trên chuỗi, mỗi `let` **sao chép** bảng biến.
3. Viết **recursive descent** trên danh sách token với **ngăn xếp phạm vi**.
4. Hiểu kỹ thuật **mỗi biến một ngăn xếp giá trị** + hoàn tác (undo) khi ra khỏi phạm vi, và tách **parse → AST → evaluate** với `ChainMap`.
5. Tránh các bẫy: gán tuần tự trong `let`, phân biệt "cặp gán" với "biểu thức cuối", số âm.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `expression` là một biểu thức hợp lệ. Trả về giá trị nguyên của nó. Ngữ pháp:

- **Số nguyên**: có thể âm, ví dụ `5`, `-3`.
- **Biến**: bắt đầu bằng chữ thường, theo sau là chữ thường hoặc chữ số, ví dụ `x`, `a1`. Các tên `add`, `let`, `mult` được bảo lưu, không bao giờ là tên biến.
- **`(let v1 e1 v2 e2 ... vn en expr)`**: gán **lần lượt** `v1 = giá trị e1`, rồi `v2 = giá trị e2` (được dùng `v1` vừa gán)…, cuối cùng trả về giá trị `expr`.
- **`(add e1 e2)`**: tổng. **`(mult e1 e2)`**: tích.

**Phạm vi:** khi tra một biến, tìm ở phạm vi **trong cùng** trước, rồi lần ra ngoài. Đề đảm bảo mọi biểu thức hợp lệ (mọi biến được dùng đều đã có giá trị).

**Ví dụ**

```text
Ví dụ 1: "(let x 2 (mult x (let x 3 y 4 (add x y))))"  -> 14
   (let x 3 y 4 (add x y)) = 3 + 4 = 7 (x bên trong che x bên ngoài), rồi 2 * 7 = 14.
Ví dụ 2: "(let x 3 x 2 x)"                              -> 2
   Gán tuần tự: x = 3 rồi x = 2.
Ví dụ 3: "(let x 1 y 2 x (add x y) (add x y))"          -> 5
   x = 1, y = 2, rồi x = (add 1 2) = 3; cuối cùng (add 3 2) = 5.
```

**Ràng buộc**

- `1 <= expression.length <= 2000`
- Không có khoảng trắng thừa; các token cách nhau đúng một dấu cách.
- Kết quả và mọi giá trị trung gian vừa kiểu int 32-bit.
- Biểu thức luôn hợp lệ.

In [1]:
EXAMPLES = [
    ("(let x 2 (mult x (let x 3 y 4 (add x y))))", 14),
    ("(let x 3 x 2 x)", 2),
    ("(let x 1 y 2 x (add x y) (add x y))", 5),
    ("(add 1 2)", 3),
    ("(mult 3 (add 2 3))", 15),
    ("(let x 2 (add (let x 3 (let x 4 x)) x))", 6),
    ("(let a1 3 b2 (add a1 1) b2)", 4),
]
for e, v in EXAMPLES:
    print(f"{v:>4}  <- {e}")

  14  <- (let x 2 (mult x (let x 3 y 4 (add x y))))
   2  <- (let x 3 x 2 x)
   5  <- (let x 1 y 2 x (add x y) (add x y))
   3  <- (add 1 2)
  15  <- (mult 3 (add 2 3))
   6  <- (let x 2 (add (let x 3 (let x 4 x)) x))
   4  <- (let a1 3 b2 (add a1 1) b2)


## 3. Kiến thức nền

### 3.1. Tách token bằng `replace` + `split`

Chèn khoảng trắng quanh ngoặc rồi `split()` — mỗi phần tử là `(`, `)`, số, biến, hoặc từ khoá.

In [2]:
def tokenize(expr):
    return expr.replace("(", " ( ").replace(")", " ) ").split()

print(tokenize("(let x 2 (mult x -3))"))

['(', 'let', 'x', '2', '(', 'mult', 'x', '-3', ')', ')']


### 3.2. Nhận diện số nguyên (kể cả số âm)

`"-3".isdigit()` là `False`! Cách an toàn: token là số nếu ký tự đầu là chữ số **hoặc** là `-`. (Tên biến luôn bắt đầu bằng chữ thường.)

In [3]:
def is_number(tok):
    return tok[0].isdigit() or tok[0] == "-"

for t in ["12", "-3", "x1", "add", "0"]:
    print(f"{t!r:6} số? {is_number(t)}   isdigit()? {t.isdigit()}")

'12'   số? True   isdigit()? True
'-3'   số? True   isdigit()? False
'x1'   số? False   isdigit()? False
'add'  số? False   isdigit()? False
'0'    số? True   isdigit()? True


### 3.3. Tách các thành phần "cấp cao nhất" trong một cặp ngoặc

Với chuỗi `"add x (mult 2 y)"`, các thành phần cấp cao nhất là `add`, `x`, `(mult 2 y)`. Ta duyệt và đếm **độ cân bằng** ngoặc: chỉ cắt ở dấu cách khi cân bằng = 0.

In [4]:
def split_top(inner):
    parts, bal, start = [], 0, 0
    for i, ch in enumerate(inner):
        if ch == "(":
            bal += 1
        elif ch == ")":
            bal -= 1
        elif ch == " " and bal == 0:
            parts.append(inner[start:i])
            start = i + 1
    parts.append(inner[start:])
    return parts

print(split_top("add x (mult 2 y)"))
print(split_top("let x 2 (mult x (let x 3 y 4 (add x y)))"))

['add', 'x', '(mult 2 y)']
['let', 'x', '2', '(mult x (let x 3 y 4 (add x y)))']


### 3.4. Phạm vi lồng nhau: list các dict, hoặc `collections.ChainMap`

`ChainMap(d_trong, d_ngoài)` tra cứu `d_trong` trước, không thấy mới sang `d_ngoài`. `cm.new_child()` tạo phạm vi con mới **không sao chép** dữ liệu; gán vào `ChainMap` chỉ ghi vào dict đầu tiên.

In [5]:
from collections import ChainMap
outer = ChainMap({"x": 2})
inner = outer.new_child()
inner["x"] = 3; inner["y"] = 4
print("trong:", inner["x"], inner["y"], " ngoài:", outer["x"], "y" in outer)

trong: 3 4  ngoài: 2 False


### 3.5. Con trỏ chia sẻ trong hàm đệ quy lồng nhau: `nonlocal`

Khi viết recursive descent bằng hàm lồng, biến vị trí `i` phải được khai báo `nonlocal` để hàm con **ghi** được vào nó.

In [6]:
def demo():
    i = 0
    def advance():
        nonlocal i
        i += 1
        return i
    advance(); advance()
    return i

print(demo())

2


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ngữ pháp

```text
expr := NUMBER | VAR
      | "(" "add"  expr expr ")"
      | "(" "mult" expr expr ")"
      | "(" "let" (VAR expr)* expr ")"
```

Giống bài *Number of Atoms*: ngữ pháp **tự tham chiếu** → đệ quy hoặc ngăn xếp.

### 4.2. Phân biệt "cặp gán" và "biểu thức cuối" trong `let`

Sau từ khoá `let`, ta đọc từng phần:

- Nếu token hiện tại là `(` hoặc là **số** → đó là biểu thức cuối.
- Nếu là **tên biến** và token **ngay sau** là `)` → biến đó là biểu thức cuối.
- Ngược lại → cặp gán `VAR expr`.

### 4.3. Tính tay ví dụ 1

`(let x 2 (mult x (let x 3 y 4 (add x y))))`

| Bước | Hành động | Phạm vi (trong cùng bên phải) |
|---|---|---|
| 1 | vào `let`, gán `x = 2` | [{x:2}] |
| 2 | gặp `(mult ...)`: là biểu thức cuối | [{x:2}] |
| 3 | `x` → 2 | [{x:2}] |
| 4 | vào `let` trong: `x = 3`, `y = 4` | [{x:2}, {x:3, y:4}] |
| 5 | `(add x y)` = 3 + 4 = 7; ra khỏi `let` trong | [{x:2}] |
| 6 | `mult` = 2 × 7 = 14 | [{x:2}] |

In [7]:
scopes = [{"x": 2}]
x_outer = scopes[-1]["x"]
scopes.append({"x": 3, "y": 4})
inner_val = scopes[-1]["x"] + scopes[-1]["y"]
scopes.pop()
print(x_outer * inner_val)

14


## 5. Cách 1 — Baseline: đệ quy trên chuỗi, sao chép bảng biến

**Ý tưởng.** Hàm `ev(expr, env)`:

- Nếu `expr` không bắt đầu bằng `(`: là số → `int`, là biến → `env[expr]`.
- Ngược lại: bỏ ngoặc ngoài, `split_top` lấy các thành phần. Với `add`/`mult` tính hai con. Với `let`: **sao chép** `env` thành `env2 = dict(env)`, gán lần lượt, trả về giá trị phần cuối.

Sao chép đảm bảo khi ra khỏi `let`, `env` của cha **không bị đổi** — đơn giản và gần như không thể sai → dùng làm **trọng tài**.

In [8]:
def evaluate_baseline(expression):
    def ev(expr, env):
        if expr[0] != "(":
            return int(expr) if is_number(expr) else env[expr]
        parts = split_top(expr[1:-1])
        op = parts[0]
        if op == "add":
            return ev(parts[1], env) + ev(parts[2], env)
        if op == "mult":
            return ev(parts[1], env) * ev(parts[2], env)
        env2 = dict(env)                         # let: phạm vi mới (sao chép)
        for k in range(1, len(parts) - 1, 2):
            env2[parts[k]] = ev(parts[k + 1], env2)
        return ev(parts[-1], env2)
    return ev(expression, {})

for e, v in EXAMPLES:
    assert evaluate_baseline(e) == v
print("Baseline đúng trên", len(EXAMPLES), "ví dụ")

Baseline đúng trên 7 ví dụ


**Độ phức tạp.** Mỗi lần gọi `ev` trên biểu thức con dài `L` tốn `O(L)` để `split_top`, và biểu thức con lồng sâu `d` tầng bị quét lại `d` lần → `O(n·d) = O(n^2)` trường hợp xấu. Mỗi `let` sao chép `env` (tới `O(số biến)`) → thêm `O(n^2)`. Với `n = 2000` vẫn qua, nhưng lãng phí.

**Nút thắt.** (1) Cắt chuỗi lặp đi lặp lại; (2) sao chép bảng biến. Giải pháp: tách token **một lần**, đọc bằng **con trỏ**; quản lý phạm vi bằng **ngăn xếp** (thêm khi vào `let`, bỏ khi ra) thay vì sao chép.

## 6. Bước đột phá

**Insight 1 — token một lần, con trỏ chạy một chiều.** Mỗi token được đọc đúng một lần ⇒ phần phân tích cú pháp `O(n)`.

**Insight 2 — phạm vi là ngăn xếp.** Các `let` lồng nhau mở/đóng theo kiểu **vào sau ra trước** ⇒ ngăn xếp. Hai cách hiện thực:

- **Ngăn xếp các dict**: vào `let` → `push({})`; gán → ghi vào dict đỉnh; tra biến → duyệt từ đỉnh xuống; ra `let` → `pop()`.
- **Mỗi biến một ngăn xếp giá trị**: `vals[name]` là list; gán trong `let` hiện tại → `append` (lần đầu trong `let` này) hoặc ghi đè đỉnh; ra `let` → `pop` các biến đã `append`. Tra biến `O(1)`: `vals[name][-1]`.

**Bất biến.** Trong khi đánh giá một biểu thức ở độ sâu `let` thứ `d`, giá trị nhìn thấy của mỗi biến là giá trị gán **gần nhất** trong các `let` đang mở 1..d.

**Quy tắc nếu… thì… (khi đọc token):**

- Nếu là số → trả `int`.
- Nếu là biến → tra phạm vi.
- Nếu là `(`: đọc từ khoá. `add`/`mult` → đọc hai biểu thức; `let` → mở phạm vi, lặp đọc cặp gán theo mục 4.2, đọc biểu thức cuối, đóng phạm vi. Cuối cùng bỏ qua `)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Recursive descent trên token + ngăn xếp các dict

**Giải thích.** `parse()` đọc một biểu thức bắt đầu tại `tokens[i]`, trả về giá trị và đẩy `i` qua biểu thức đó.

In [9]:
def evaluate_scope_stack(expression):
    tokens = tokenize(expression)
    i = 0
    scopes = [{}]

    def lookup(name):
        for sc in reversed(scopes):
            if name in sc:
                return sc[name]
        raise KeyError(name)

    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1
            return int(tok) if is_number(tok) else lookup(tok)
        i += 1                                   # bỏ '('
        op = tokens[i]; i += 1
        if op == "add" or op == "mult":
            a = parse(); b = parse()
            i += 1                               # bỏ ')'
            return a + b if op == "add" else a * b
        scopes.append({})                        # let
        while True:
            tok = tokens[i]
            if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                val = parse()                    # biểu thức cuối
                break
            i += 1                               # tên biến
            scopes[-1][tok] = parse()
        i += 1                                   # bỏ ')'
        scopes.pop()
        return val

    return parse()

for e, v in EXAMPLES:
    assert evaluate_scope_stack(e) == v
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


**Code nộp LeetCode (Cách 2).**

```python
class Solution:
    def evaluate(self, expression: str) -> int:
        tokens = expression.replace('(', ' ( ').replace(')', ' ) ').split()
        self.i = 0
        scopes = [{}]

        def is_num(t):
            return t[0].isdigit() or t[0] == '-'

        def lookup(name):
            for sc in reversed(scopes):
                if name in sc:
                    return sc[name]

        def parse():
            tok = tokens[self.i]
            if tok != '(':
                self.i += 1
                return int(tok) if is_num(tok) else lookup(tok)
            self.i += 1
            op = tokens[self.i]
            self.i += 1
            if op in ('add', 'mult'):
                a = parse()
                b = parse()
                self.i += 1
                return a + b if op == 'add' else a * b
            scopes.append({})
            while True:
                tok = tokens[self.i]
                if tok == '(' or is_num(tok) or tokens[self.i + 1] == ')':
                    val = parse()
                    break
                self.i += 1
                scopes[-1][tok] = parse()
            self.i += 1
            scopes.pop()
            return val

        return parse()
```

**Độ phức tạp.** Phân tích `O(n)`; mỗi lần tra biến tốn `O(d)` (số phạm vi lồng). Tổng `O(n·d)` trường hợp xấu, thực tế nhỏ. Space `O(n)`.

### 7.2. Cách 3 — Mỗi biến một ngăn xếp giá trị (tra cứu `O(1)`)

**Giải thích.** `vals = defaultdict(list)`. Trong mỗi `let`, giữ tập `declared` các biến đã `append` trong `let` này:

- Gán `v = val`: nếu `v` chưa ở `declared` → `vals[v].append(val)`, thêm vào `declared`; ngược lại → `vals[v][-1] = val` (gán lại trong cùng `let`, như ví dụ 2).
- Ra khỏi `let`: với mỗi `v` trong `declared` → `vals[v].pop()`.
- Tra biến: `vals[v][-1]`.

In [10]:
from collections import defaultdict

def evaluate_var_stacks(expression):
    tokens = tokenize(expression)
    i = 0
    vals = defaultdict(list)

    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1
            return int(tok) if is_number(tok) else vals[tok][-1]
        i += 1
        op = tokens[i]; i += 1
        if op in ("add", "mult"):
            a = parse(); b = parse(); i += 1
            return a + b if op == "add" else a * b
        declared = set()
        while True:
            tok = tokens[i]
            if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                val = parse()
                break
            i += 1
            v = parse()                           # tính TRƯỚC khi gán (vế phải dùng giá trị cũ)
            if tok in declared:
                vals[tok][-1] = v
            else:
                vals[tok].append(v)
                declared.add(tok)
        i += 1
        for name in declared:
            vals[name].pop()
        return val

    return parse()

for e, v in EXAMPLES:
    assert evaluate_var_stacks(e) == v
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


**Code nộp LeetCode (Cách 3).**

```python
from collections import defaultdict

class Solution:
    def evaluate(self, expression: str) -> int:
        tokens = expression.replace('(', ' ( ').replace(')', ' ) ').split()
        vals = defaultdict(list)
        self.i = 0

        def is_num(t):
            return t[0].isdigit() or t[0] == '-'

        def parse():
            tok = tokens[self.i]
            if tok != '(':
                self.i += 1
                return int(tok) if is_num(tok) else vals[tok][-1]
            self.i += 1
            op = tokens[self.i]
            self.i += 1
            if op in ('add', 'mult'):
                a = parse()
                b = parse()
                self.i += 1
                return a + b if op == 'add' else a * b
            declared = set()
            while True:
                tok = tokens[self.i]
                if tok == '(' or is_num(tok) or tokens[self.i + 1] == ')':
                    val = parse()
                    break
                self.i += 1
                v = parse()
                if tok in declared:
                    vals[tok][-1] = v
                else:
                    vals[tok].append(v)
                    declared.add(tok)
            self.i += 1
            for name in declared:
                vals[name].pop()
            return val

        return parse()
```

**Độ phức tạp.** Mỗi token xử lý `O(1)` (tra cứu, gán, hoàn tác đều `O(1)` trung bình) ⇒ **`O(n)`** time, `O(n)` space.

### 7.3. Cách 4 — Parse thành cây (AST), rồi evaluate với `ChainMap`

**Giải thích.** Tách hai pha giống trình biên dịch thật:

1. **Parse**: token → cây. Nút là tuple: `("num", 5)`, `("var", "x")`, `("add", A, B)`, `("mult", A, B)`, `("let", [(v1, E1), ...], body)`.
2. **Evaluate**: duyệt cây; `let` tạo `env.new_child()`.

Ưu điểm: cây có thể in ra, tối ưu, hoặc đánh giá nhiều lần với môi trường khác nhau. Pha parse không cần biết gì về giá trị.

In [11]:
def parse_ast(expression):
    tokens = tokenize(expression)
    i = 0

    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1
            return ("num", int(tok)) if is_number(tok) else ("var", tok)
        i += 1
        op = tokens[i]; i += 1
        if op in ("add", "mult"):
            a = parse(); b = parse(); i += 1
            return (op, a, b)
        binds = []
        while True:
            tok = tokens[i]
            if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                body = parse()
                break
            i += 1
            binds.append((tok, parse()))
        i += 1
        return ("let", binds, body)

    return parse()

def eval_ast(node, env):
    kind = node[0]
    if kind == "num":
        return node[1]
    if kind == "var":
        return env[node[1]]
    if kind == "add":
        return eval_ast(node[1], env) + eval_ast(node[2], env)
    if kind == "mult":
        return eval_ast(node[1], env) * eval_ast(node[2], env)
    child = env.new_child()
    for name, e in node[1]:
        child[name] = eval_ast(e, child)
    return eval_ast(node[2], child)

def evaluate_ast(expression):
    return eval_ast(parse_ast(expression), ChainMap())

print(parse_ast("(let x 2 (mult x (let x 3 y 4 (add x y))))"))
for e, v in EXAMPLES:
    assert evaluate_ast(e) == v
print("Cách 4 đúng trên ví dụ")

('let', [('x', ('num', 2))], ('mult', ('var', 'x'), ('let', [('x', ('num', 3)), ('y', ('num', 4))], ('add', ('var', 'x'), ('var', 'y')))))
Cách 4 đúng trên ví dụ


**Code nộp LeetCode (Cách 4).**

```python
from collections import ChainMap

class Solution:
    def evaluate(self, expression: str) -> int:
        tokens = expression.replace('(', ' ( ').replace(')', ' ) ').split()
        self.i = 0

        def is_num(t):
            return t[0].isdigit() or t[0] == '-'

        def parse():
            tok = tokens[self.i]
            if tok != '(':
                self.i += 1
                return ('num', int(tok)) if is_num(tok) else ('var', tok)
            self.i += 1
            op = tokens[self.i]
            self.i += 1
            if op in ('add', 'mult'):
                a = parse()
                b = parse()
                self.i += 1
                return (op, a, b)
            binds = []
            while True:
                tok = tokens[self.i]
                if tok == '(' or is_num(tok) or tokens[self.i + 1] == ')':
                    body = parse()
                    break
                self.i += 1
                binds.append((tok, parse()))
            self.i += 1
            return ('let', binds, body)

        def ev(node, env):
            k = node[0]
            if k == 'num':
                return node[1]
            if k == 'var':
                return env[node[1]]
            if k == 'add':
                return ev(node[1], env) + ev(node[2], env)
            if k == 'mult':
                return ev(node[1], env) * ev(node[2], env)
            child = env.new_child()
            for name, e in node[1]:
                child[name] = ev(e, child)
            return ev(node[2], child)

        return ev(parse(), ChainMap())
```

**Độ phức tạp.** Parse `O(n)`. Evaluate: tra `ChainMap` tốn `O(d)` → `O(n·d)` trường hợp xấu. Space `O(n)` cho cây.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 với ví dụ 3: `(let x 1 y 2 x (add x y) (add x y))`

| Bước | Token đang xử lý | Hành động | `vals` |
|---|---|---|---|
| 1 | `x 1` | x chưa khai báo trong let → append | {x:[1]} |
| 2 | `y 2` | append | {x:[1], y:[2]} |
| 3 | `x (add x y)` | tính (add 1 2)=3; x đã khai báo → ghi đè đỉnh | {x:[3], y:[2]} |
| 4 | `(add x y)` | biểu thức cuối = 3 + 2 = 5 | |
| 5 | `)` | pop x, y | {x:[], y:[]} |

In [12]:
def trace_var_stacks(expression):
    tokens = tokenize(expression)
    i = 0
    vals = defaultdict(list)
    def show():
        return {k: v[:] for k, v in vals.items() if v}
    def parse(depth=0):
        nonlocal i
        pad = "  " * depth
        tok = tokens[i]
        if tok != "(":
            i += 1
            return int(tok) if is_number(tok) else vals[tok][-1]
        i += 1
        op = tokens[i]; i += 1
        if op in ("add", "mult"):
            a = parse(depth + 1); b = parse(depth + 1); i += 1
            r = a + b if op == "add" else a * b
            print(f"{pad}({op} {a} {b}) = {r}")
            return r
        print(f"{pad}vào let")
        declared = set()
        while True:
            tok = tokens[i]
            if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                val = parse(depth + 1); break
            i += 1
            v = parse(depth + 1)
            if tok in declared:
                vals[tok][-1] = v; how = "ghi đè"
            else:
                vals[tok].append(v); declared.add(tok); how = "push"
            print(f"{pad}  {tok} = {v} ({how}) -> {show()}")
        i += 1
        for name in declared:
            vals[name].pop()
        print(f"{pad}ra let, giá trị {val}; pop {sorted(declared)} -> {show()}")
        return val
    return parse()

print("Kết quả:", trace_var_stacks("(let x 1 y 2 x (add x y) (add x y))"))

vào let
  x = 1 (push) -> {'x': [1]}
  y = 2 (push) -> {'x': [1], 'y': [2]}
  (add 1 2) = 3
  x = 3 (ghi đè) -> {'x': [3], 'y': [2]}
  (add 3 2) = 5
ra let, giá trị 5; pop ['x', 'y'] -> {}
Kết quả: 5


In [13]:
print("Kết quả:", trace_var_stacks("(let x 2 (mult x (let x 3 y 4 (add x y))))"))

vào let
  x = 2 (push) -> {'x': [2]}
    vào let
      x = 3 (push) -> {'x': [2, 3]}
      y = 4 (push) -> {'x': [2, 3], 'y': [4]}
      (add 3 4) = 7
    ra let, giá trị 7; pop ['x', 'y'] -> {'x': [2]}
  (mult 2 7) = 14
ra let, giá trị 14; pop ['x'] -> {}
Kết quả: 14


### 8.2. Trace con trỏ token cho `(let x 3 x 2 x)`

Token: `['(', 'let', 'x', '3', 'x', '2', 'x', ')']`, chỉ số 0..7.

- `i=0` `(` → `i=1` `let` → `i=2`.
- `i=2`: `x`, token sau là `3` (không phải `)`) → cặp gán; đọc `3` → `x=3`, `i=4`.
- `i=4`: `x`, token sau là `2` → cặp gán `x=2`, `i=6`.
- `i=6`: `x`, token sau là `)` → biểu thức cuối → giá trị 2, `i=7`; bỏ `)` → `i=8`.

In [14]:
tokens = tokenize("(let x 3 x 2 x)")
for k, t in enumerate(tokens):
    nxt = tokens[k + 1] if k + 1 < len(tokens) else None
    print(k, repr(t), "| token sau:", repr(nxt))

0 '(' | token sau: 'let'
1 'let' | token sau: 'x'
2 'x' | token sau: '3'
3 '3' | token sau: 'x'
4 'x' | token sau: '2'
5 '2' | token sau: 'x'
6 'x' | token sau: ')'
7 ')' | token sau: None


## 9. Code nộp LeetCode

Bản chính: **Cách 3 (mỗi biến một ngăn xếp)** — `O(n)`, gọn.

In [15]:
from collections import defaultdict

class Solution:
    def evaluate(self, expression: str) -> int:
        tokens = expression.replace('(', ' ( ').replace(')', ' ) ').split()
        vals = defaultdict(list)
        self.i = 0

        def is_num(t):
            return t[0].isdigit() or t[0] == '-'

        def parse():
            tok = tokens[self.i]
            if tok != '(':
                self.i += 1
                return int(tok) if is_num(tok) else vals[tok][-1]
            self.i += 1
            op = tokens[self.i]
            self.i += 1
            if op in ('add', 'mult'):
                a = parse()
                b = parse()
                self.i += 1
                return a + b if op == 'add' else a * b
            declared = set()
            while True:
                tok = tokens[self.i]
                if tok == '(' or is_num(tok) or tokens[self.i + 1] == ')':
                    val = parse()
                    break
                self.i += 1
                v = parse()
                if tok in declared:
                    vals[tok][-1] = v
                else:
                    vals[tok].append(v)
                    declared.add(tok)
            self.i += 1
            for name in declared:
                vals[name].pop()
            return val

        return parse()

for e, v in EXAMPLES:
    assert Solution().evaluate(e) == v
print("Solution OK trên ví dụ")

Solution OK trên ví dụ


## 10. Vì sao thuật toán đúng

**Phân tích cú pháp đúng.** Ngữ pháp ở 4.1 là **LL(2)**: nhìn token hiện tại và token kế tiếp là đủ quyết định dùng luật nào. Trong `let`:

- `(` hoặc số không thể là tên biến → không thể bắt đầu cặp gán → chắc chắn là biểu thức cuối.
- Tên biến theo sau bởi `)` → nếu là cặp gán thì phải có `expr` trước `)` — mâu thuẫn → là biểu thức cuối.
- Tên biến theo sau bởi token khác `)` → nếu là biểu thức cuối thì token kế phải là `)` — mâu thuẫn → là cặp gán.

Mỗi hàm `parse()` đọc **đúng** các token của một biểu thức (quy nạp theo độ dài), nên con trỏ luôn ở đúng vị trí.

**Ngữ nghĩa phạm vi đúng (Cách 3).** Bất biến: tại mọi thời điểm, `vals[v][-1]` là giá trị gán gần nhất của `v` trong các `let` **đang mở**. Vào `let` không đổi gì; gán lần đầu trong `let` → push (che giá trị ngoài); gán lại trong cùng `let` → ghi đè đỉnh (chính là phần tử của `let` này, vì mọi `let` con đã pop sạch phần của mình trước khi trả về); ra khỏi `let` → pop đúng các phần tử đã push → khôi phục trạng thái khi vào. Vế phải được tính **trước** khi ghi nên `(let x 1 x (add x 1) x)` dùng `x = 1` cũ → 2.

**Cách 1, 2, 4** đúng vì cùng mô hình: phạm vi mới khi vào `let`, gán tuần tự vào phạm vi đó, tra từ trong ra ngoài.

## 11. Phân tích Time/Space

`n` = độ dài chuỗi, `d` = độ sâu lồng `let` tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Chuỗi + sao chép env | `O(n^2)` | `O(n^2)` xấu nhất |
| 2. Token + stack dict | `O(n·d)` | `O(n)` |
| 3. Stack theo biến | `O(n)` | `O(n)` |
| 4. AST + ChainMap | `O(n·d)` | `O(n)` |

Đệ quy sâu tối đa ~`n/4` tầng (mỗi tầng tốn ≥ 4 ký tự như `(add`) ≈ 500 → nên tăng recursion limit cho chắc.

In [16]:
import sys, time, random
sys.setrecursionlimit(10000)

def deep_let(depth):
    # (let x 1 (let x (add x 1) (let x (add x 1) ... x)))
    s = "(let x 1 "
    for _ in range(depth):
        s += "(let x (add x 1) "
    return s + "x" + ")" * (depth + 1)

e = deep_let(150)
print("độ dài", len(e))
for fn in [evaluate_baseline, evaluate_scope_stack, evaluate_var_stacks, evaluate_ast, Solution().evaluate]:
    t = time.perf_counter(); r = fn(e); t = time.perf_counter() - t
    print(f"{getattr(fn, '__name__', ''):<22} {t*1000:7.2f} ms -> {r}")

độ dài 2711
evaluate_baseline        11.59 ms -> 151
evaluate_scope_stack      0.15 ms -> 151
evaluate_var_stacks       0.14 ms -> 151
evaluate_ast              0.56 ms -> 151
evaluate                  0.17 ms -> 151


## 12. Bộ test

Ta sinh biểu thức **ngẫu nhiên hợp lệ**: chỉ dùng biến đã có trong phạm vi, có `let` lồng, gán lại, số âm. Đối chiếu mọi cách với baseline.

In [17]:
ALL = [evaluate_baseline, evaluate_scope_stack, evaluate_var_stacks, evaluate_ast, Solution().evaluate]

EDGE = [
    ("5", 5),
    ("-7", -7),
    ("(add -2 3)", 1),
    ("(mult -2 -3)", 6),
    ("(let x -2 y x y)", -2),
    ("(let x 1 x (add x 1) x)", 2),                 # vế phải dùng x cũ
    ("(let x 7 -12)", -12),                         # biểu thức cuối là số âm
    ("(let a 1 b 2 (let a 10 (add a b)))", 12),
    ("(let x 2 (add (let x 3 x) x))", 5),           # x ngoài sống lại
    ("(let var1 4 var2 (mult var1 var1) var2)", 16),
]
for fn in ALL:
    for e, v in EXAMPLES + EDGE:
        assert fn(e) == v, (fn, e, fn(e), v)
print("Ví dụ + edge case OK")

Ví dụ + edge case OK


In [18]:
NAMES = ["x", "y", "z", "a1", "b2"]

def rand_expr(rng, scope, depth):
    r = rng.random()
    if depth == 0 or r < 0.25:
        if scope and rng.random() < 0.6:
            return rng.choice(sorted(scope))
        return str(rng.randint(-5, 5))
    if r < 0.6:
        op = rng.choice(["add", "mult"])
        return f"({op} {rand_expr(rng, scope, depth - 1)} {rand_expr(rng, scope, depth - 1)})"
    parts, new_scope = [], set(scope)
    for _ in range(rng.randint(1, 3)):
        name = rng.choice(NAMES)
        parts.append(f"{name} {rand_expr(rng, new_scope, depth - 1)}")
        new_scope.add(name)
    return f"(let {' '.join(parts)} {rand_expr(rng, new_scope, depth - 1)})"

rng = random.Random(736)
for _ in range(3000):
    e = rand_expr(rng, set(), rng.randint(1, 5))
    ref = evaluate_baseline(e)
    for fn in ALL[1:]:
        assert fn(e) == ref, (fn, e, fn(e), ref)
print("3000 biểu thức ngẫu nhiên khớp baseline. Ví dụ:", e, "=", ref)

3000 biểu thức ngẫu nhiên khớp baseline. Ví dụ: (let b2 (let y (let z -4 b2 z b2) (let a1 5 z a1 5)) y (let z (let b2 b2 x -4 x x -1) z (let y -4 z y x b2 4) z z (add b2 -3)) (let b2 (let y b2 b2 0 b2 b2 1) b2 (let y b2 b2 4 -1) x (let b2 y x 1 y) (let z -4 b2 z b2 z 4))) = 4


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `isdigit()` để nhận diện số

**Code sai:** `int(tok) if tok.isdigit() else env[tok]` → `-3` bị coi là biến → `KeyError`.

**Code đúng:** `tok[0].isdigit() or tok[0] == '-'`.

In [19]:
def ev_isdigit_wrong(tok, env):
    return int(tok) if tok.isdigit() else env[tok]

try:
    ev_isdigit_wrong("-3", {})
except KeyError as ex:
    print("Lỗi KeyError:", ex)
print("đúng:", int("-3") if is_number("-3") else None)

Lỗi KeyError: '-3'
đúng: -3


### Lỗi 2 — Không hoàn tác phạm vi khi ra khỏi `let`

**Code sai:** dùng **một** dict toàn cục, gán thẳng vào đó.

**Vì sao sai:** `(let x 2 (add (let x 3 x) x))` → sau `let` trong, `x` vẫn là 3 → ra 6 thay vì 5.

**Code đúng:** ngăn xếp phạm vi / sao chép / undo.

In [20]:
def evaluate_global_dict_wrong(expression):
    tokens = tokenize(expression); i = 0; env = {}
    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1; return int(tok) if is_number(tok) else env[tok]
        i += 1; op = tokens[i]; i += 1
        if op in ("add", "mult"):
            a = parse(); b = parse(); i += 1
            return a + b if op == "add" else a * b
        while True:
            tok = tokens[i]
            if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                val = parse(); break
            i += 1; env[tok] = parse()            # SAI: không có phạm vi
        i += 1
        return val
    return parse()

e = "(let x 2 (add (let x 3 x) x))"
print("sai:", evaluate_global_dict_wrong(e), " đúng:", Solution().evaluate(e))

sai: 6  đúng: 5


### Lỗi 3 — Gán "song song" thay vì tuần tự

**Code sai:** tính mọi vế phải trong phạm vi **cha**, rồi mới gán hết.

**Vì sao sai:** `(let x 1 y (add x 1) y)` cần `x = 1` khi tính `y` → nếu tính trong phạm vi cha (chưa có `x`) sẽ lỗi; `(let x 3 x 2 x)` cũng phụ thuộc thứ tự.

**Code đúng:** gán **lần lượt**, mỗi vế phải tính trong phạm vi **mới** (đã có các biến trước).

In [21]:
def evaluate_parallel_wrong(expression):
    def ev(expr, env):
        if expr[0] != "(":
            return int(expr) if is_number(expr) else env[expr]
        parts = split_top(expr[1:-1])
        if parts[0] == "add":
            return ev(parts[1], env) + ev(parts[2], env)
        if parts[0] == "mult":
            return ev(parts[1], env) * ev(parts[2], env)
        new_vals = {parts[k]: ev(parts[k + 1], env) for k in range(1, len(parts) - 1, 2)}   # SAI
        return ev(parts[-1], {**env, **new_vals})
    return ev(expression, {})

e = "(let x 1 y (add x 1) y)"
try:
    print(evaluate_parallel_wrong(e))
except KeyError as ex:
    print("sai: KeyError", ex, "| đúng:", Solution().evaluate(e))

sai: KeyError 'x' | đúng: 2


### Lỗi 4 — Nhầm biến cuối cùng là cặp gán

**Code sai:** trong `let`, luôn đọc theo cặp đến khi gặp `)` → biểu thức `(let x 2 x)` đọc `x` rồi cố đọc vế phải là `)`.

**Code đúng:** kiểm tra token kế tiếp: tên biến + `)` ⇒ biểu thức cuối.

In [22]:
def let_parts_wrong(tokens_inside):
    # đọc theo cặp đến hết, phần lẻ cuối bị bỏ quên
    return [(tokens_inside[k], tokens_inside[k + 1]) for k in range(0, len(tokens_inside) - 1, 2)]

print("sai:", let_parts_wrong(["x", "2", "x"]), "-> mất biểu thức cuối 'x'")

sai: [('x', '2')] -> mất biểu thức cuối 'x'


## 14. Mẹo & nhận dạng pattern

- **Bài "viết interpreter/parser"** → tokenize một lần, recursive descent với con trỏ. Cùng họ: *Basic Calculator I–III (224, 227, 772)*, *Number of Atoms (726)*, *Decode String (394)*, *Ternary Expression Parser (439)*.
- **Phạm vi lồng nhau** → ngăn xếp (list dict / ChainMap / stack theo biến + undo).
- **Quyết định luật ngữ pháp** → nhìn trước 1–2 token (lookahead).
- **Undo log**: ghi lại những gì đã thay đổi trong phạm vi, hoàn tác khi thoát — kỹ thuật dùng lại trong backtracking và DSU rollback.
- Viết **baseline sao chép** trước làm trọng tài; tối ưu sau.

## 15. Biến thể

1. Thêm phép `sub` / `div` (chia lấy phần nguyên về 0): chỉ thêm nhánh.
2. Thêm `(if cond a b)`: đánh giá `cond`, chọn một nhánh (chú ý **không** đánh giá nhánh còn lại nếu có tác dụng phụ).
3. `add`/`mult` nhận **nhiều** đối số: `(add 1 2 3 4)`.
4. Ngăn xếp tường minh (không đệ quy) cho biểu thức cực sâu.
5. Báo lỗi biến chưa khai báo thay vì giả định hợp lệ.

In [23]:
# Biến thể 2 + 3: thêm if và add/mult nhiều đối số (dùng AST)
def parse_ast_ext(expression):
    tokens = tokenize(expression); i = 0
    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1
            return ("num", int(tok)) if is_number(tok) else ("var", tok)
        i += 1; op = tokens[i]; i += 1
        if op == "let":
            binds = []
            while True:
                tok = tokens[i]
                if tok == "(" or is_number(tok) or tokens[i + 1] == ")":
                    body = parse(); break
                i += 1; binds.append((tok, parse()))
            i += 1
            return ("let", binds, body)
        args = []
        while tokens[i] != ")":
            args.append(parse())
        i += 1
        return (op, args)
    return parse()

def eval_ext(node, env):
    k = node[0]
    if k == "num": return node[1]
    if k == "var": return env[node[1]]
    if k == "let":
        child = env.new_child()
        for name, e in node[1]:
            child[name] = eval_ext(e, child)
        return eval_ext(node[2], child)
    args = node[1]
    if k == "if":
        return eval_ext(args[1], env) if eval_ext(args[0], env) != 0 else eval_ext(args[2], env)
    vals = [eval_ext(a, env) for a in args]
    if k == "add": return sum(vals)
    if k == "mult":
        p = 1
        for v in vals: p *= v
        return p
    if k == "sub": return vals[0] - sum(vals[1:])
    raise ValueError(k)

for e in ["(add 1 2 3 4)", "(let x 0 (if x 10 20))", "(let x 5 (if (sub x 5) 1 (mult 2 3 x)))"]:
    print(e, "=", eval_ext(parse_ast_ext(e), ChainMap()))

(add 1 2 3 4) = 10
(let x 0 (if x 10 20)) = 20
(let x 5 (if (sub x 5) 1 (mult 2 3 x))) = 30


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi + sao chép | `split_top` + `dict(env)` mỗi `let` | `O(n^2)` | `O(n^2)` | Trọng tài, viết nhanh |
| 2. Token + stack dict | Recursive descent, tra từ đỉnh xuống | `O(n·d)` | `O(n)` | Dễ hiểu, phổ biến |
| 3. Stack theo biến | push/ghi đè/pop, tra `O(1)` | `O(n)` | `O(n)` | Bản nộp tối ưu |
| 4. AST + ChainMap | Tách parse và evaluate | `O(n·d)` | `O(n)` | Muốn mở rộng ngôn ngữ |

## 17. Bài tập

**Bài 1 (đoán output).** `evaluate("(let x 2 y (mult x 3) (let x (add y 1) (add x y)))")` = ?

**Bài 2 (tính tay).** Lập bảng `vals` (như 8.1) cho `(let a 1 (let b a a 5 (add a b)))`.

**Bài 3 (điền TODO).** Hoàn thiện `is_let_body(tokens, i)` trả về `True` nếu `tokens[i]` bắt đầu biểu thức cuối của `let`.

**Bài 4 (tự viết lại).** Viết lại Cách 2 (stack các dict) trong hàm `my_evaluate(expression)`.

**Bài 5 (mini challenge).** Viết `free_vars(expression)` trả về tập các biến được **dùng** mà **không** được khai báo trong phạm vi nào bao quanh (với biểu thức có thể không hợp lệ). Ví dụ `(let x 1 (add x y))` → `{"y"}`.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def is_let_body(tokens, i):
    # TODO: '(' hoặc số -> True; tên biến và tokens[i+1] == ')' -> True; còn lại False
    pass

# Bài 4 — TODO
def my_evaluate(expression):
    # TODO: tokenize; scopes = [{}]; parse() đệ quy với nonlocal i
    pass

# Bài 5 — TODO
def free_vars(expression):
    # TODO: dùng parse_ast, duyệt cây với tập 'bound'; gặp var không thuộc bound -> thêm
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `x = 2`, `y = 6`.
- Mức 2: `let` trong: `x = y + 1 = 7`.
- Mức 3: `(add 7 6) = 13`.

**Bài 2.**
- Mức 1: `a` push 1 → {a:[1]}.
- Mức 2: `let` trong: `b = a = 1` (push) → {a:[1], b:[1]}; `a = 5` (push, vì `a` chưa khai báo trong `let` trong) → {a:[1,5], b:[1]}.
- Mức 3: `(add 5 1) = 6`; ra `let` trong pop `a`, `b` → {a:[1]}. Kết quả **6**.

**Bài 3.**
- Mức 1: dùng `is_number`.
- Mức 2: `tokens[i] == '('`.
- Mức 3: `tokens[i + 1] == ')'`.

**Bài 4.**
- Mức 1: `lookup` duyệt `reversed(scopes)`.
- Mức 2: `let`: `scopes.append({})` trước, `pop()` sau.
- Mức 3: đừng quên `i += 1` để bỏ `)`.

**Bài 5.**
- Mức 1: duyệt AST đệ quy, mang theo `bound` (set).
- Mức 2: trong `let`, vế phải của cặp thứ `k` thấy các biến của cặp `< k`.
- Mức 3: biểu thức cuối thấy mọi biến của `let`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
assert Solution().evaluate("(let x 2 y (mult x 3) (let x (add y 1) (add x y)))") == 13
assert Solution().evaluate("(let a 1 (let b a a 5 (add a b)))") == 6

# Bài 3
def is_let_body_ans(tokens, i):
    t = tokens[i]
    return t == "(" or is_number(t) or tokens[i + 1] == ")"

tk = tokenize("(let x 3 x 2 x)")
assert [is_let_body_ans(tk, k) for k in (2, 4, 6)] == [False, False, True]
print("Bài 1–3 OK")

Bài 1–3 OK


In [26]:
# Bài 4
def my_evaluate_ans(expression):
    tokens = tokenize(expression); i = 0; scopes = [{}]
    def parse():
        nonlocal i
        tok = tokens[i]
        if tok != "(":
            i += 1
            if is_number(tok):
                return int(tok)
            for sc in reversed(scopes):
                if tok in sc:
                    return sc[tok]
        i += 1; op = tokens[i]; i += 1
        if op in ("add", "mult"):
            a = parse(); b = parse(); i += 1
            return a + b if op == "add" else a * b
        scopes.append({})
        while not is_let_body_ans(tokens, i):
            name = tokens[i]; i += 1
            scopes[-1][name] = parse()
        val = parse(); i += 1
        scopes.pop()
        return val
    return parse()

# Bài 5
def free_vars_ans(expression):
    res = set()
    def walk(node, bound):
        k = node[0]
        if k == "num":
            return
        if k == "var":
            if node[1] not in bound:
                res.add(node[1])
            return
        if k in ("add", "mult"):
            walk(node[1], bound); walk(node[2], bound); return
        b = set(bound)
        for name, e in node[1]:
            walk(e, b)
            b.add(name)
        walk(node[2], b)
    walk(parse_ast(expression), set())
    return res

rng = random.Random(42)
for _ in range(500):
    e = rand_expr(rng, set(), 4)
    assert my_evaluate_ans(e) == evaluate_baseline(e)
    assert free_vars_ans(e) == set()           # biểu thức sinh ra luôn hợp lệ
assert free_vars_ans("(let x 1 (add x y))") == {"y"}
assert free_vars_ans("(let x y y 2 (add x y))") == {"y"}
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template recursive descent + phạm vi:**

```python
tokens = s.replace('(', ' ( ').replace(')', ' ) ').split()
def parse():
    tok = tokens[i]
    if tok != '(':  i += 1; return số hoặc tra biến
    i += 1; op = tokens[i]; i += 1
    if op in ('add', 'mult'): a = parse(); b = parse(); i += 1; return ...
    mở phạm vi
    while not (tokens[i] == '(' or is_num(tokens[i]) or tokens[i+1] == ')'):
        name = tokens[i]; i += 1; gán name = parse()   # tuần tự
    val = parse(); i += 1; đóng phạm vi; return val
```

**Checklist trước khi submit:**

- [ ] Số âm: kiểm tra `tok[0] == '-'`.
- [ ] Phân biệt cặp gán / biểu thức cuối bằng lookahead.
- [ ] Gán **tuần tự**, vế phải tính trước khi ghi.
- [ ] Gán lại cùng biến trong một `let`: ghi đè, không push thêm.
- [ ] Ra khỏi `let` phải khôi phục phạm vi.
- [ ] Bỏ qua `)` sau mỗi biểu thức ngoặc.
- [ ] Đệ quy sâu ~500: ổn, có thể tăng recursion limit.